In [ ]:
import numpy as np
from copy import deepcopy
import matplotlib.pyplot as plt
from matplotlib import cm
from matplotlib.colors import Normalize

from pypower.api import runopf, ppoption
from pypower.idx_gen import PG, QG, PMIN, PMAX

# -----------------------------
# WB5 case in Python / PYPOWER format
# -----------------------------
def make_WB5():
    mpc = {
        "version": "2",
        "baseMVA": 100.0,
        "bus": np.array([
            [1, 3,   0,   0, 0, 0, 1, 1.0,    0,   345, 1, 1.05, 0.95],
            [2, 1, 130,  20, 0, 0, 1, 1.0,  -10,   345, 1, 1.05, 0.95],
            [3, 1, 130,  20, 0, 0, 1, 1.0,  -20,   345, 1, 1.05, 0.95],
            [4, 1,  65,  10, 0, 0, 1, 1.0, -135,   345, 1, 1.05, 0.95],
            [5, 2,   0,   0, 0, 0, 1, 1.0, -140,   345, 1, 1.05, 0.95],
        ], dtype=float),
        "gen": np.array([
            [1, 500, 50, 1800, -30, 1, 100, 1, 5000, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
            [5,   0,  0, 1800, -30, 1, 100, 1, 5000, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        ], dtype=float),
        "branch": np.array([
            [1, 2, 0.04, 0.09, 0.00, 2500, 2500, 2500, 0, 0, 1, -360, 360],
            [1, 3, 0.05, 0.10, 0.00, 2500, 2500, 2500, 0, 0, 1, -360, 360],
            [2, 4, 0.55, 0.90, 0.45, 2500, 2500, 2500, 0, 0, 1, -360, 360],
            [3, 5, 0.55, 0.90, 0.45, 2500, 2500, 2500, 0, 0, 1, -360, 360],
            [4, 5, 0.06, 0.10, 0.00, 2500, 2500, 2500, 0, 0, 1, -360, 360],
            [2, 3, 0.07, 0.09, 0.00, 2500, 2500, 2500, 0, 0, 1, -360, 360],
        ], dtype=float),
        "areas": np.array([[1, 5]], dtype=float),
        "gencost": np.array([
            [2, 2, 0, 3, 0, 4.00, 0],
            [2, 2, 0, 3, 0, 1.00, 0],
        ], dtype=float),
    }
    return mpc

mpc0 = make_WB5()

ppopt = ppoption(VERBOSE=0, OUT_ALL=0)
res = runopf(mpc0, ppopt)
print("OPF success:", bool(res["success"]))
print("Objective:", res["f"])

OPF success: True
Objective: 1082.3325036854237


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
from matplotlib.colors import Normalize
from copy import deepcopy

from pypower.api import runopf, ppoption
from pypower.idx_gen import PG, QG, PMIN, PMAX

def solve_fixed_pg(pg1_mw, pg5_mw, base_case, ppopt):
    case = deepcopy(base_case)

    # Fix generator active powers by setting Pmin = Pmax = Pg
    case["gen"][0, PG] = pg1_mw
    case["gen"][0, PMIN] = pg1_mw
    case["gen"][0, PMAX] = pg1_mw

    case["gen"][1, PG] = pg5_mw
    case["gen"][1, PMIN] = pg5_mw
    case["gen"][1, PMAX] = pg5_mw

    r = runopf(case, ppopt)
    if not r["success"]:
        return None

    qg5_pu = r["gen"][1, QG] / r["baseMVA"]
    cost = r["f"]
    return qg5_pu, cost

# Grid in MW
Pg1_vals = np.linspace(0, 5000, 45)
Pg5_vals = np.linspace(0, 5000, 45)

Z = np.full((len(Pg5_vals), len(Pg1_vals)), np.nan)   # Q_G5 in pu
C = np.full((len(Pg5_vals), len(Pg1_vals)), np.nan)   # objective cost

for i, pg5 in enumerate(Pg5_vals):
    for j, pg1 in enumerate(Pg1_vals):
        out = solve_fixed_pg(pg1, pg5, mpc0, ppopt)
        if out is not None:
            qg5_pu, cost = out
            Z[i, j] = qg5_pu
            C[i, j] = cost

# Mesh for plotting
P1g, P5g = np.meshgrid(Pg1_vals / mpc0["baseMVA"], Pg5_vals / mpc0["baseMVA"])

fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection="3d")

# Feasible surface colored by cost
norm = Normalize(vmin=np.nanmin(C), vmax=np.nanmax(C))
facecolors = cm.Reds(norm(C))
facecolors[np.isnan(Z)] = (0, 0, 0, 0)

surf = ax.plot_surface(
    P1g, P5g, Z,
    facecolors=facecolors,
    linewidth=0,
    antialiased=True,
    shade=False
)

# Gray lower reactive power limit plane, like the paper
Q_plane = -0.30 * np.ones_like(Z)
ax.plot_surface(
    P1g, P5g, Q_plane,
    color="lightgray",
    alpha=0.55,
    linewidth=0,
    shade=False
)

# Example markers; replace with your actual global/local optimum points
# ax.scatter(pg1_star, pg5_star, qg5_star, marker='*', s=220, color='limegreen', edgecolor='k')
# ax.scatter(pg1_local, pg5_local, qg5_local, marker='v', s=120, color='royalblue', edgecolor='k')

ax.set_xlabel(r"$P_{G1}$ (per unit)", labelpad=12)
ax.set_ylabel(r"$P_{G5}$ (per unit)", labelpad=12)
ax.set_zlabel(r"$Q_{G5}$ (per unit)", labelpad=12)
ax.view_init(elev=12, azim=-135)

mappable = cm.ScalarMappable(norm=norm, cmap=cm.Reds)
mappable.set_array([])
cbar = fig.colorbar(mappable, ax=ax, pad=0.08, shrink=0.9)
cbar.set_label("Generation cost")
plt.tight_layout()
plt.show()